# Gaussian Class

## Motivation

A multivariate Gaussian is represented as

$$
\mathbf{X}\sim\mathcal{N}(\boldsymbol{\mu},\Sigma).
$$

This notebook builds a reusable `Gaussian` class for the **Representation** chapter. It contains only operations intrinsic to the distribution itself:

- parameter validation,
- dimension,
- probability density,
- log-density,
- Mahalanobis distance,
- random sampling.

Inference-specific methods such as `marginal()`, `condition()`, and `multiply()` are intentionally postponed until their mathematics is studied in the inference chapter.


In [32]:
from __future__ import annotations

from dataclasses import dataclass

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import numpy as np


## 1. Class Structure and Validation

The mean must be a one-dimensional vector,

$$
\boldsymbol{\mu}\in\mathbb{R}^d,
$$

and the covariance must be a symmetric, positive-semidefinite matrix,

$$
\Sigma\in\mathbb{R}^{d\times d}.
$$

In [33]:
@dataclass
class Gaussian:
    """
    Multivariate Gaussian distribution.

    X ~ N(mean, covariance)
    """

    mean: np.ndarray
    covariance: np.ndarray

    def __post_init__(self) -> None:
        self.mean = np.asarray(self.mean, dtype=float)
        self.covariance = np.asarray(
            self.covariance,
            dtype=float,
        )
        self._validate_parameters()

    @property
    def dimension(self) -> int:
        return self.mean.size

    def _validate_parameters(self) -> None:
        if self.mean.ndim != 1:
            raise ValueError(
                "Mean must be a one-dimensional vector."
            )

        if self.mean.size == 0:
            raise ValueError(
                "Mean vector cannot be empty."
            )

        if self.covariance.shape != (
            self.dimension,
            self.dimension,
        ):
            raise ValueError(
                "Covariance must have shape "
                "(dimension, dimension)."
            )

        if not np.allclose(
            self.covariance,
            self.covariance.T,
        ):
            raise ValueError(
                "Covariance matrix must be symmetric."
            )

        eigenvalues = np.linalg.eigvalsh(
            self.covariance
        )

        if np.any(eigenvalues < -1e-10):
            raise ValueError(
                "Covariance matrix must be "
                "positive semidefinite."
            )

    def mahalanobis_distance(
        self,
        x: np.ndarray,
    ) -> float:
        x = np.asarray(x, dtype=float)

        if x.shape != self.mean.shape:
            raise ValueError(
                "x must have the same shape as mean."
            )

        difference = x - self.mean

        distance_squared = difference @ np.linalg.solve(
            self.covariance,
            difference,
        )

        return float(np.sqrt(distance_squared))

    def log_pdf(
        self,
        x: np.ndarray,
    ) -> float:
        x = np.asarray(x, dtype=float)

        if x.shape != self.mean.shape:
            raise ValueError(
                "x must have the same shape as mean."
            )

        sign, log_determinant = np.linalg.slogdet(
            self.covariance
        )

        if sign <= 0:
            raise ValueError(
                "Covariance must be positive definite "
                "to evaluate the density."
            )

        mahalanobis_squared = (
            self.mahalanobis_distance(x) ** 2
        )

        return float(
            -0.5
            * (
                self.dimension * np.log(2.0 * np.pi)
                + log_determinant
                + mahalanobis_squared
            )
        )

    def pdf(
        self,
        x: np.ndarray,
    ) -> float:
        return float(np.exp(self.log_pdf(x)))

    def sample(
        self,
        n_samples: int | None = None,
        rng: np.random.Generator | None = None,
    ) -> np.ndarray:
        if rng is None:
            rng = np.random.default_rng()

        if n_samples is not None and n_samples <= 0:
            raise ValueError(
                "n_samples must be positive."
            )

        return rng.multivariate_normal(
            mean=self.mean,
            cov=self.covariance,
            size=n_samples,
        )

    def _validate_2d_plot(self) -> None:
        if self.dimension != 2:
            raise ValueError(
                "Visualization is currently supported only "
                "for two-dimensional Gaussians."
            )


    def _pdf_grid(
        self,
        num_points: int = 100,
        span: float = 4.0,
    ) -> tuple[
        np.ndarray,
        np.ndarray,
        np.ndarray,
    ]:
        self._validate_2d_plot()
    
        if num_points <= 1:
            raise ValueError(
                "num_points must be greater than 1."
            )
    
        if span <= 0:
            raise ValueError(
                "span must be positive."
            )
    
        # Scale plotting range using standard deviation
        standard_deviation = np.sqrt(
            np.diag(self.covariance)
        )
    
        x1 = np.linspace(
            self.mean[0]
            - span * standard_deviation[0],
            self.mean[0]
            + span * standard_deviation[0],
            num_points,
        )
    
        x2 = np.linspace(
            self.mean[1]
            - span * standard_deviation[1],
            self.mean[1]
            + span * standard_deviation[1],
            num_points,
        )
    
        X1, X2 = np.meshgrid(
            x1,
            x2,
        )
    
        points = np.column_stack([
            X1.ravel(),
            X2.ravel(),
        ])
    
        density = np.array([
            self.pdf(point)
            for point in points
        ])
    
        density = density.reshape(
            X1.shape
        )
    
        return X1, X2, density

    def plot_samples(
        self,
        n_samples: int = 2000,
        rng: np.random.Generator | None = None,
    ) -> None:
        self._validate_2d_plot()
    
        samples = self.sample(
            n_samples=n_samples,
            rng=rng,
        )
    
        plt.figure(
            figsize=(7, 6)
        )
    
        plt.scatter(
            samples[:, 0],
            samples[:, 1],
            alpha=0.25,
        )
    
        plt.scatter(
            self.mean[0],
            self.mean[1],
            marker="x",
            s=120,
            label="Mean",
        )
    
        plt.xlabel("$X_1$")
        plt.ylabel("$X_2$")
    
        plt.title(
            "Samples from a Multivariate Gaussian"
        )
    
        plt.axis("equal")
        plt.grid(alpha=0.25)
        plt.legend()
    
        plt.show()

    def plot_pdf_2d(
        self,
        num_points: int = 100,
        span: float = 4.0,
        levels: int = 10,
    ) -> None:
        X1, X2, density = self._pdf_grid(
            num_points=num_points,
            span=span,
        )
    
        plt.figure(
            figsize=(7, 6)
        )
    
        contour = plt.contour(
            X1,
            X2,
            density,
            levels=levels,
        )
    
        plt.clabel(
            contour,
            inline=True,
            fontsize=8,
        )
    
        plt.scatter(
            self.mean[0],
            self.mean[1],
            marker="x",
            s=120,
            label="Mean",
        )
    
        plt.xlabel("$X_1$")
        plt.ylabel("$X_2$")
    
        plt.title(
            "2D Gaussian Probability Density"
        )
    
        plt.axis("equal")
        plt.grid(alpha=0.25)
        plt.legend()
    
        plt.show()

    def plot_pdf_3d(
        self,
        num_points: int = 100,
        span: float = 4.0,
        width: int = 1000,
        height: int = 800,
    ) -> None:
        X1, X2, density = self._pdf_grid(
            num_points=num_points,
            span=span,
        )
    
        figure = go.Figure(
            data=[
                go.Surface(
                    x=X1,
                    y=X2,
                    z=density,
                )
            ]
        )
    
        figure.update_layout(
            title="3D Gaussian Probability Density",
            width=width,
            height=height,
            scene={
                "xaxis_title": "X₁",
                "yaxis_title": "X₂",
                "zaxis_title": "p(X₁, X₂)",
            },
        )
    
        figure.show()

## 2. Basic Test

In [45]:
#gaussian = Gaussian(
#    mean=np.array([2.0, 3.0]),
#    covariance=np.array([
#        [3.0, 1.5],
#        [1.5, 2.0],
#    ]),
#)

#gaussian.plot_pdf_3d()
